# Materia oscura como red de entrelazamiento (ER = EPR): prueba con curvas de rotación

**Proyecto:** extensión de *Vacuum Particle Journey*.

Este notebook compara, galaxia a galaxia y con estimación de error, cinco explicaciones de las curvas de rotación:

| Modelo | Idea | Parámetros libres |
|---|---|---|
| **Solo bariones** | sin materia oscura (referencia) | $\Upsilon_\star$ |
| **NFW** | halo de materia oscura fría (ΛCDM) | $\Upsilon_\star, V_{200}, c$ |
| **Verlinde (2016)** | gravedad emergente: entrelazamiento del vacío de De Sitter | $\Upsilon_\star$ (nada más: $a_0 = cH_0$ es fijo) |
| **Red ER (juguete)** | nuestra hipótesis: la materia se entrelaza con su entorno y los puentes ER aportan energía | $\Upsilon_\star, \kappa, \lambda$ |
| **Yukawa** | una fuerza nueva atractiva/repulsiva con alcance $\lambda$ (mediador con masa) | $\Upsilon_\star, \alpha, \lambda$ |

**Datos:** catálogo SPARC (Lelli, McGaugh y Schombert 2016), con 175 galaxias de disco. Si no se puede descargar, el notebook usa una galaxia sintética para comprobar que todo funciona.

**Errores:** $\chi^2$, $\chi^2$ reducido, AIC/BIC (que penalizan los parámetros extra), incertidumbres de los parámetros por la matriz de covarianza y un *bootstrap* paramétrico.

> Ejecuta las celdas en orden (*Entorno de ejecución → Ejecutar todas*). Solo se usan numpy, scipy, matplotlib y pandas, que ya vienen en Colab.

In [ ]:
import os, io, re, glob, zipfile, urllib.request, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.special import i0, i1, k0, k1
from scipy.integrate import cumulative_trapezoid

warnings.filterwarnings("ignore", category=RuntimeWarning)
RNG = np.random.default_rng(42)

# Colores: uno por modelo, siempre el mismo en todas las figuras
COL = {
    "Datos": "#222222",
    "Solo bariones": "#8a8a85",
    "NFW": "#2a78d6",
    "Verlinde": "#eb6834",
    "Red ER (juguete)": "#1baf7a",
    "Yukawa": "#6250d6",
}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "lines.linewidth": 2, "font.size": 10,
    "legend.frameon": False,
})

## 1. Constantes y unidades

Trabajamos en kpc, km/s y masas solares. Así $v^2$ está en (km/s)² y $G M / r$ también.

In [ ]:
G = 4.30091e-6            # kpc (km/s)^2 / Msun
C_KMS = 299792.458        # km/s
H0 = 70.0                 # km/s/Mpc
H0_KPC = H0 / 1000.0      # km/s/kpc
A0_VERLINDE = C_KMS * H0_KPC   # a0 = c*H0 en (km/s)^2/kpc
KPC_M = 3.0857e19         # metros por kpc
HBARC_EVM = 1.97327e-7    # hbar*c en eV*m

print(f"a0 = c*H0 = {A0_VERLINDE:.0f} (km/s)^2/kpc = {A0_VERLINDE*1e6/KPC_M:.2e} m/s^2")
print(f"a0/6      = {A0_VERLINDE*1e6/KPC_M/6:.2e} m/s^2   (la escala MOND observada es ~1.2e-10 m/s^2)")

## 2. Antes de ajustar: partículas sin masa, carga y fuerza fuerte

La idea de que partículas **sin masa pero con carga**, y con **fuerza nuclear fuerte**, generen la atracción/repulsión que falta tiene sentido como pregunta. Hay tres hechos que la condicionan:

1. **Las únicas partículas sin masa con carga son los gluones**, que llevan *carga de color*. Los fotones no tienen masa pero tampoco carga. Los gluones están **confinados**: la fuerza fuerte no se extiende más allá de ~1 fm ($10^{-15}$ m). Fuera del protón, lo que queda es la fuerza nuclear residual, transmitida por piones, con alcance ~1.4 fm.
2. **Una fuerza transmitida por un mediador de masa $m$ decae como Yukawa:** $F \propto e^{-r/\lambda}/r^2$, con $\lambda = \hbar/(mc)$. Para notarse a escala de galaxias ($\lambda \sim$ kpc), el mediador tendría que ser absurdamente ligero.
3. **Las partículas sin masa viajan a $c$** y no pueden quedarse ligadas a una galaxia (velocidad de escape ~500 km/s). Por eso la materia oscura tiene que ser "fría".

La celda siguiente pone números a esto. Aun así, la idea no se descarta: en el **universo temprano** (plasma de quarks y gluones) la fuerza fuerte generó muchísimo entrelazamiento. Ese entrelazamiento es justo lo que en ER = EPR se convierte en geometría. Es decir, la fuerza fuerte puede ser el **origen** de la red de puentes, aunque no sea la fuerza que actúa hoy a escala galáctica. Por eso más abajo ajustamos también un modelo **Yukawa** con $\alpha$ (que puede ser positiva o negativa) y $\lambda$ libres: los datos dirán qué alcance haría falta.

In [ ]:
r_kpc_m = 1.0 * KPC_M
fuerzas = {
    "Fuerte (gluones, confinamiento)": 1.0e-15,
    "Nuclear residual (piones)":       1.4e-15,
    "Débil (bosones W/Z)":             2.5e-18,
}
print("Supresión Yukawa e^{-r/λ} a r = 1 kpc:")
for nombre, lam in fuerzas.items():
    log10_sup = -(r_kpc_m / lam) / np.log(10)
    print(f"  {nombre:34s} λ = {lam:.1e} m  ->  e^(-r/λ) = 10^({log10_sup:.2e})")

print("\nMasa del mediador necesaria para que una fuerza alcance λ:")
for lam_kpc in [1, 10, 100]:
    m_eV = HBARC_EVM / (lam_kpc * KPC_M)
    print(f"  λ = {lam_kpc:4d} kpc  ->  m c^2 = {m_eV:.1e} eV   (un pión pesa 1.4e8 eV)")

v_esc = 550.0  # km/s, Vía Láctea cerca del Sol
print(f"\nPartículas sin masa: v = c = {C_KMS:.0f} km/s, unas {C_KMS/v_esc:.0f} veces la velocidad de escape galáctica.")

## 3. Datos: SPARC

Cada archivo `*_rotmod.dat` trae, para cada radio $r$: la velocidad observada $V_{obs}$ con su error, y las contribuciones del gas, el disco estelar y el bulbo ($V_{gas}, V_{disk}, V_{bul}$). Estas dos últimas están calculadas con relación masa-luz $\Upsilon = 1$. La velocidad bariónica es

$$v_b^2 = V_{gas}|V_{gas}| + \Upsilon_d\, V_{disk}|V_{disk}| + \Upsilon_b\, V_{bul}|V_{bul}|, \qquad \Upsilon_b = 1.4\,\Upsilon_d .$$

Si la descarga automática falla, Colab te pedirá subir `Rotmod_LTG.zip`, que puedes bajar a mano de la web de SPARC. Si tampoco hay archivo, se usa una galaxia sintética.

In [ ]:
DATA_DIR = "sparc_rotmod"
URLS = [
    "https://astroweb.case.edu/SPARC/Rotmod_LTG.zip",
    "http://astroweb.cwru.edu/SPARC/Rotmod_LTG.zip",
    "https://zenodo.org/records/16284118/files/Rotmod_LTG.zip?download=1",
]

def _files():
    return sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_rotmod.dat"), recursive=True))

def get_sparc():
    os.makedirs(DATA_DIR, exist_ok=True)
    if _files():
        return True
    for url in URLS:
        try:
            print("Descargando", url)
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            data = urllib.request.urlopen(req, timeout=60).read()
            zipfile.ZipFile(io.BytesIO(data)).extractall(DATA_DIR)
            if _files():
                return True
        except Exception as e:
            print("  falló:", type(e).__name__, e)
    try:
        from google.colab import files
        print("Sube Rotmod_LTG.zip (web de SPARC: astroweb.case.edu/SPARC)")
        for name, content in files.upload().items():
            zipfile.ZipFile(io.BytesIO(content)).extractall(DATA_DIR)
        return bool(_files())
    except Exception:
        return False

def load_rotmod(path):
    name = os.path.basename(path).replace("_rotmod.dat", "")
    dist = np.nan
    with open(path) as f:
        for line in f:
            m = re.search(r"Distance\s*=\s*([\d.]+)", line)
            if m:
                dist = float(m.group(1)); break
    d = np.atleast_2d(np.loadtxt(path, comments="#"))
    g = dict(name=name, dist=dist, r=d[:, 0], vobs=d[:, 1], err=d[:, 2],
             vgas=d[:, 3], vdisk=d[:, 4], vbul=d[:, 5])
    ok = (g["r"] > 0) & (g["err"] > 0)
    for k in ["r", "vobs", "err", "vgas", "vdisk", "vbul"]:
        g[k] = g[k][ok]
    return g

# --- Galaxia sintética (para validar el código): disco exponencial + gas + halo NFW conocido ---
def v2_expdisk(r, M, Rd):
    y = r / (2 * Rd)
    Sigma0 = M / (2 * np.pi * Rd**2)
    return 4 * np.pi * G * Sigma0 * Rd * y**2 * (i0(y) * k0(y) - i1(y) * k1(y))

def make_synthetic(V200=150.0, c=10.0, ups=0.5, seed=1):
    rng = np.random.default_rng(seed)
    r = np.linspace(0.5, 30, 35)
    vdisk = np.sqrt(v2_expdisk(r, 6e10, 3.0))   # con Υ = 1 (6e10 Lsun)
    vgas = np.sqrt(v2_expdisk(r, 1.2e10, 8.0))
    vbul = np.zeros_like(r)
    vb2 = vgas**2 + ups * vdisk**2
    vtrue = np.sqrt(vb2 + v2_nfw(r, V200, c))
    err = 3.0 + 0.03 * vtrue
    vobs = vtrue + rng.normal(0, err)
    return dict(name="SINTETICA_NFW", dist=10.0, r=r, vobs=vobs, err=err,
                vgas=vgas, vdisk=vdisk, vbul=vbul, truth=dict(V200=V200, c=c, ups=ups))

## 4. Los modelos

**NFW** (halo de materia oscura fría), con $x = r/R_{200}$ y $R_{200} = V_{200}/(10 H_0)$:
$$v_{NFW}^2 = V_{200}^2\,\frac{\mu(cx)}{x\,\mu(c)},\qquad \mu(y)=\ln(1+y)-\frac{y}{1+y}$$

**Verlinde**: con $G M_B(r) = r\,v_b^2$ (aproximación esférica),
$$M_D^2(r) = \frac{a_0 r^2}{6G}\frac{d(M_B r)}{dr}\;\;\Rightarrow\;\; v_D^2 = \sqrt{\frac{a_0}{6}\,\frac{d\,(r^2 v_b^2)}{dr}}$$
No tiene ningún parámetro libre aparte de $\Upsilon_\star$.

**Red ER (juguete)**: cada elemento de masa bariónica está entrelazado con su entorno, y la energía de los puentes ER se reparte alrededor según un núcleo normalizado de alcance $\lambda$:
$$\rho_{ER}(\mathbf{x}) = \kappa\int \rho_b(\mathbf{y})\,K(|\mathbf{x}-\mathbf{y}|)\,d^3y,\qquad K(d)=\frac{e^{-d/\lambda}}{4\pi\lambda^2 d},\quad \int K\,d^3d = 1$$
$\kappa$ es la energía de la red por unidad de masa bariónica. Para un cascarón esférico de radio $r'$, el promedio angular del núcleo es analítico:
$$\langle K\rangle(r,r') = \frac{1}{8\pi\lambda r r'}\Big[e^{-|r-r'|/\lambda}-e^{-(r+r')/\lambda}\Big]$$
Luego $M_{ER}(r)=\int_0^r 4\pi s^2\rho_{ER}\,ds$ y $v_{ER}^2 = G M_{ER}/r$.

**Yukawa**: una fuerza nueva entre bariones, $F = \frac{GMm}{r^2}\big[1+\alpha(1+r/\lambda)e^{-r/\lambda}\big]$. Con $\alpha>0$ es atractiva y con $\alpha<0$ repulsiva. Usamos la masa encerrada como si fuera puntual (aproximación):
$$v^2 = v_b^2\big[1+\alpha(1+r/\lambda)e^{-r/\lambda}\big]$$

In [ ]:
def vbar2(g, ud):
    ub = 1.4 * ud
    return (g["vgas"] * np.abs(g["vgas"]) + ud * g["vdisk"] * np.abs(g["vdisk"])
            + ub * g["vbul"] * np.abs(g["vbul"]))

def v2_nfw(r, V200, c):
    R200 = V200 / (10 * H0_KPC)
    x = r / R200
    mu = lambda y: np.log1p(y) - y / (1 + y)
    return V200**2 * mu(c * x) / (x * mu(c))

def GM_baryon(r, vb2):
    # G*M_b(r), forzando que la masa encerrada no decrezca
    return np.maximum.accumulate(r * np.clip(vb2, 0, None))

def v2_verlinde_extra(r, vb2):
    y = r * GM_baryon(r, vb2)                # = G * M_B * r
    dy = np.gradient(y, r)
    return np.sqrt(A0_VERLINDE / 6.0 * np.clip(dy, 0, None))

def v2_er_extra(r, vb2, kappa, lam, n_grid=200):
    GM = GM_baryon(r, vb2)
    rg = np.linspace(r.max() / n_grid, r.max(), n_grid)
    GMg = np.interp(rg, np.r_[0.0, r], np.r_[0.0, GM])
    dGM = np.diff(np.r_[0.0, GMg])            # G * masa de cada cascarón
    S, R = rg[:, None], rg[None, :]
    kern = (np.exp(-np.abs(S - R) / lam) - np.exp(-(S + R) / lam)) / (8 * np.pi * lam * S * R)
    Grho = kappa * kern @ dGM                 # G * rho_ER
    GM_er = cumulative_trapezoid(4 * np.pi * rg**2 * Grho, rg, initial=0.0)
    return np.interp(r, rg, GM_er) / r

def v2_yukawa(r, vb2, alpha, lam):
    return vb2 * (1 + alpha * (1 + r / lam) * np.exp(-r / lam))

MODELS = {
    "Solo bariones": dict(names=["Υ_d"], p0=[0.5], lo=[0.1], hi=[1.5],
        v2=lambda g, p: vbar2(g, p[0])),
    "NFW": dict(names=["Υ_d", "V200 [km/s]", "c"], p0=[0.5, 120, 10], lo=[0.1, 10, 1], hi=[1.5, 500, 50],
        v2=lambda g, p: vbar2(g, p[0]) + v2_nfw(g["r"], p[1], p[2])),
    "Verlinde": dict(names=["Υ_d"], p0=[0.5], lo=[0.1], hi=[1.5],
        v2=lambda g, p: (lambda vb2: vb2 + v2_verlinde_extra(g["r"], vb2))(vbar2(g, p[0]))),
    "Red ER (juguete)": dict(names=["Υ_d", "κ", "λ [kpc]"], p0=[0.5, 3, 10], lo=[0.1, 0, 0.2], hi=[1.5, 100, 300],
        v2=lambda g, p: (lambda vb2: vb2 + v2_er_extra(g["r"], vb2, p[1], p[2]))(vbar2(g, p[0]))),
    "Yukawa": dict(names=["Υ_d", "α", "λ [kpc]"], p0=[0.5, 1, 10], lo=[0.1, -0.99, 0.1], hi=[1.5, 50, 300],
        v2=lambda g, p: v2_yukawa(g["r"], vbar2(g, p[0]), p[1], p[2])),
}

# Comprobación del modelo ER: con λ muy pequeño, M_ER ≈ κ·M_b (la red sigue a la materia)
_r = np.linspace(0.5, 20, 40); _vb2 = 100**2 * np.ones_like(_r)
_ratio = v2_er_extra(_r, _vb2, kappa=1.0, lam=0.2, n_grid=600) / _vb2
print("Test ER (λ→0): v_ER²/v_b² en r = 10 kpc ≈", round(float(np.interp(10, _r, _ratio)), 3), "(debería ser ≈ 1)")

## 5. Ajuste y estimación de error

- **Residuo:** $(V_{obs}-V_{mod})/\sigma$ en cada radio, con $\chi^2=\sum$ residuo².
- **Prior sobre $\Upsilon_d$:** log-normal centrada en 0.5 con 0.1 dex, como en la literatura de SPARC. Evita que un modelo "haga trampa" inflando la masa estelar. Se puede desactivar.
- **$\chi^2_\nu = \chi^2/(N-k)$:** ≈1 es un buen ajuste; ≫1 indica que el modelo no describe los datos.
- **AIC $= \chi^2 + 2k$, BIC $= \chi^2 + k\ln N$:** penalizan los parámetros extra. Una diferencia $\Delta$BIC > 6 es evidencia fuerte a favor del modelo con menor BIC.
- **Incertidumbres:** $\mathrm{Cov} = (J^TJ)^{-1}\cdot\max(\chi^2_\nu,1)$ y, aparte, un *bootstrap* paramétrico que simula datos nuevos con el mejor ajuste y su error, y vuelve a ajustar.

Se prueban varios puntos de partida para no quedarse en un mínimo local.

In [ ]:
USE_UPSILON_PRIOR = True

def model_v(g, name, p):
    return np.sqrt(np.clip(MODELS[name]["v2"](g, p), 0, None))

def residuals(p, g, name):
    res = (g["vobs"] - model_v(g, name, p)) / g["err"]
    if USE_UPSILON_PRIOR:
        res = np.r_[res, np.log10(p[0] / 0.5) / 0.1]
    return res

def fit_model(g, name, n_starts=6, rng=RNG):
    m = MODELS[name]
    lo, hi = np.array(m["lo"], float), np.array(m["hi"], float)
    starts = [np.clip(np.array(m["p0"], float), lo, hi)]
    starts += [lo + rng.random(len(lo)) * (hi - lo) for _ in range(n_starts - 1)]
    best = None
    for p0 in starts:
        try:
            r = least_squares(residuals, p0, bounds=(lo, hi), args=(g, name), method="trf")
        except Exception:
            continue
        if best is None or r.cost < best.cost:
            best = r
    p = best.x
    v = model_v(g, name, p)
    chi2 = float(np.sum(((g["vobs"] - v) / g["err"])**2))
    N, k = len(g["r"]), len(p)
    dof = max(N - k, 1)
    J = best.jac
    cov = np.linalg.pinv(J.T @ J) * max(chi2 / dof, 1.0)
    perr = np.sqrt(np.clip(np.diag(cov), 0, None))
    return dict(model=name, p=p, perr=perr, chi2=chi2, chi2r=chi2 / dof,
                aic=chi2 + 2 * k, bic=chi2 + k * np.log(N), k=k, N=N, v=v)

def fit_all(g, n_starts=6):
    return {name: fit_model(g, name, n_starts) for name in MODELS}

def table(fits):
    rows = []
    best_bic = min(f["bic"] for f in fits.values())
    for name, f in fits.items():
        pars = ", ".join(f"{n}={v:.3g}±{e:.2g}" for n, v, e in zip(MODELS[name]["names"], f["p"], f["perr"]))
        rows.append(dict(Modelo=name, k=f["k"], chi2=round(f["chi2"], 1), chi2_red=round(f["chi2r"], 2),
                         AIC=round(f["aic"], 1), BIC=round(f["bic"], 1), dBIC=round(f["bic"] - best_bic, 1),
                         Parametros=pars))
    return pd.DataFrame(rows).set_index("Modelo")

## 6. Validación con una galaxia sintética

Fabricamos una galaxia con un halo NFW conocido ($V_{200}=150$, $c=10$, $\Upsilon=0.5$) y ruido. Si el código funciona, NFW debería ganar y recuperar esos valores dentro de sus errores.

In [ ]:
gsyn = make_synthetic()
fits_syn = fit_all(gsyn)
print("Valores verdaderos:", gsyn["truth"])
table(fits_syn)

## 7. Una galaxia real en detalle

Por defecto, **NGC 3198**, la curva de rotación plana clásica. Cambia `GALAXY` para probar otra: `NGC2403`, `DDO154`, `UGC02885`, `NGC6503`…

In [ ]:
HAVE_SPARC = get_sparc()
GALAXIES = {}
if HAVE_SPARC:
    for f in _files():
        g = load_rotmod(f)
        if len(g["r"]) >= 5:
            GALAXIES[g["name"]] = g
    print(f"Cargadas {len(GALAXIES)} galaxias de SPARC.")
else:
    print("Sin datos SPARC: se usará solo la galaxia sintética.")

GALAXY = "NGC3198"
g = GALAXIES.get(GALAXY, gsyn) if HAVE_SPARC else gsyn
print("Galaxia:", g["name"], f"| D = {g['dist']} Mpc | {len(g['r'])} puntos")
fits = fit_all(g)
table(fits)

In [ ]:
def plot_galaxy(g, fits):
    fig, (ax, axr) = plt.subplots(2, 1, figsize=(8.5, 7), sharex=True,
                                  gridspec_kw=dict(height_ratios=[3, 1.3], hspace=0.06))
    r = g["r"]
    ax.errorbar(r, g["vobs"], g["err"], fmt="o", ms=4, color=COL["Datos"], ecolor="#9a9a95",
                elinewidth=1, capsize=0, label="Observado", zorder=5)
    f0 = fits["Solo bariones"]
    ax.plot(r, f0["v"], "--", color=COL["Solo bariones"], label=f"Solo bariones (χ²ν={f0['chi2r']:.1f})")
    for name in ["NFW", "Verlinde", "Red ER (juguete)", "Yukawa"]:
        f = fits[name]
        ax.plot(r, f["v"], color=COL[name], label=f"{name} (χ²ν={f['chi2r']:.1f})")
        axr.plot(r, (g["vobs"] - f["v"]) / g["err"], "o-", ms=3, lw=1, color=COL[name])
    ax.set_ylabel("Velocidad de rotación [km/s]")
    ax.set_title(f"{g['name']}: curva de rotación y modelos", loc="left")
    ax.set_ylim(0, None)
    ax.legend(loc="lower right", fontsize=9)
    axr.axhline(0, color="#c3c2b7", lw=1)
    axr.set_ylabel("Residuo [σ]")
    axr.set_xlabel("Radio [kpc]")
    plt.show()

plot_galaxy(g, fits)

### Bootstrap paramétrico

Generamos `N_BOOT` curvas simuladas $V_{sim} = V_{mejor\,ajuste} + \mathcal{N}(0,\sigma)$, ajustamos cada una y vemos cuánto varían los parámetros. Si el intervalo 16–84 % es mucho más ancho que el error de la covarianza, el parámetro está mal determinado (degeneración). Ocurre a menudo con $\kappa$ y $\lambda$ del modelo ER.

In [ ]:
N_BOOT = 60   # súbelo a 200+ para resultados finales (tarda más)

def bootstrap(g, name, fit, n=N_BOOT):
    samples = []
    for _ in range(n):
        gs = dict(g); gs["vobs"] = fit["v"] + RNG.normal(0, g["err"])
        samples.append(fit_model(gs, name, n_starts=3)["p"])
    return np.array(samples)

rows = []
t0 = time.time()
for name in ["NFW", "Verlinde", "Red ER (juguete)", "Yukawa"]:
    S = bootstrap(g, name, fits[name])
    for j, pn in enumerate(MODELS[name]["names"]):
        lo_, med, hi_ = np.percentile(S[:, j], [16, 50, 84])
        rows.append(dict(Modelo=name, Parametro=pn, Mejor=round(fits[name]["p"][j], 3),
                         Error_cov=round(fits[name]["perr"][j], 3),
                         Boot_p16=round(lo_, 3), Boot_p50=round(med, 3), Boot_p84=round(hi_, 3)))
print(f"Bootstrap en {time.time()-t0:.0f} s")
pd.DataFrame(rows)

## 8. Todas las galaxias de SPARC

Ajustamos los 5 modelos a cada galaxia (tarda unos minutos) y resumimos:
- la mediana de $\chi^2_\nu$ de cada modelo,
- en qué fracción de galaxias gana cada modelo según el BIC,
- la distribución de $\chi^2_\nu$.

In [ ]:
def fit_sample(galaxies, n_starts=4):
    out = []
    t0 = time.time()
    for i, (name, gg) in enumerate(galaxies.items()):
        fs = fit_all(gg, n_starts)
        best = min(fs, key=lambda m: fs[m]["bic"])
        for m, f in fs.items():
            out.append(dict(galaxia=name, modelo=m, chi2r=f["chi2r"], bic=f["bic"], gana=(m == best),
                            **{f"p{j}": v for j, v in enumerate(f["p"])}))
        if (i + 1) % 25 == 0:
            print(f"  {i+1}/{len(galaxies)} galaxias, {time.time()-t0:.0f} s")
    return pd.DataFrame(out)

sample = GALAXIES if HAVE_SPARC else {"SINTETICA_NFW": gsyn}
res = fit_sample(sample)
resumen = res.groupby("modelo").agg(chi2r_mediana=("chi2r", "median"),
                                    galaxias_con_chi2r_menor_2=("chi2r", lambda x: f"{np.mean(x < 2):.0%}"),
                                    gana_por_BIC=("gana", lambda x: f"{np.mean(x):.0%}"))
resumen.loc[list(MODELS)]

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))
bins = np.linspace(-1, 3, 41)
for m in MODELS:
    x = np.log10(res.loc[res.modelo == m, "chi2r"].clip(lower=1e-1))
    ls = "--" if m == "Solo bariones" else "-"
    ax.hist(x, bins=bins, histtype="step", lw=2, ls=ls, color=COL[m], label=m)
ax.axvline(0, color="#c3c2b7", lw=1)
ax.set_xlabel("log10 χ²ν   (0 = ajuste perfecto)")
ax.set_ylabel("Número de galaxias")
ax.set_title("Calidad de ajuste en toda la muestra", loc="left")
ax.legend(fontsize=9)
plt.show()

## 9. Prueba sin parámetros: relación de Tully–Fisher bariónica

Verlinde predice, sin ajustar nada, que en la zona plana $v_f^4 = G M_b\,a_0/6$. Comparamos con los datos: $v_f$ es la media de los 3 últimos puntos y $M_b$ la masa bariónica con $\Upsilon_d=0.5$. Un halo NFW no predice esta relación por sí mismo; tiene que salir de cómo se formaron las galaxias.

In [ ]:
if HAVE_SPARC:
    Mb, vf = [], []
    for gg in GALAXIES.values():
        if len(gg["r"]) < 6:
            continue
        vb2 = vbar2(gg, 0.5)
        Mb.append(GM_baryon(gg["r"], vb2)[-1] / G)
        vf.append(np.mean(gg["vobs"][-3:]))
    Mb, vf = np.array(Mb), np.array(vf)
    ok = Mb > 0
    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.loglog(Mb[ok], vf[ok], "o", ms=4, color=COL["Datos"], alpha=0.7, label="SPARC")
    mm = np.logspace(np.log10(Mb[ok].min()), np.log10(Mb[ok].max()), 50)
    ax.loglog(mm, (G * mm * A0_VERLINDE / 6) ** 0.25, color=COL["Verlinde"], label="Verlinde: v⁴ = G M a₀/6")
    ax.set_xlabel("Masa bariónica [M☉]"); ax.set_ylabel("v plana [km/s]")
    ax.set_title("Tully–Fisher bariónica", loc="left"); ax.legend()
    plt.show()
    resid = np.log10(vf[ok]) - 0.25 * np.log10(G * Mb[ok] * A0_VERLINDE / 6)
    print(f"Desviación media: {resid.mean():+.3f} dex, dispersión: {resid.std():.3f} dex")
else:
    print("Necesita los datos SPARC.")

## 10. ¿Qué alcance tendría la fuerza "atracción/repulsión"?

Si el modelo Yukawa ajusta bien, su $\lambda$ nos dice qué masa tendría el mediador. Comparamos con la fuerza fuerte.

In [ ]:
yk = res[res.modelo == "Yukawa"]
lam = yk["p2"].values; alpha = yk["p1"].values
m_eV = HBARC_EVM / (lam * KPC_M)
print(f"λ mediana = {np.median(lam):.1f} kpc  ->  masa del mediador ~ {np.median(m_eV):.1e} eV")
print(f"α mediana = {np.median(alpha):+.2f}  ({np.mean(alpha > 0):.0%} de galaxias prefieren atracción extra)")
print(f"Galaxias que tocan el límite λ = 300 kpc: {np.mean(lam > 290):.0%}  (señal de que un alcance finito no basta)")
print(f"Comparación: el mediador de la fuerza nuclear (pión) pesa 1.4e8 eV, unas {1.4e8/np.median(m_eV):.0e} veces más.")

er = res[res.modelo == "Red ER (juguete)"]
print(f"\nRed ER: κ mediana = {np.median(er['p1']):.2f} (energía de la red por unidad de masa bariónica), "
      f"λ mediana = {np.median(er['p2']):.1f} kpc")

## 11. Cómo leer los resultados y siguientes pasos

**Cómo interpretar:**
- Si **Verlinde** queda cerca de NFW en $\chi^2_\nu$ teniendo 2 parámetros menos, es un resultado notable para las ideas de "entrelazamiento → gravedad".
- Si la **red ER** ajusta bien pero $\kappa$ y $\lambda$ cambian mucho de una galaxia a otra, el modelo es flexible pero no predictivo. Una teoría seria debería fijar $\kappa$ y $\lambda$ a partir de primeros principios (por ejemplo, $\lambda \sim c/H_0$ o la escala de decoherencia).
- Si **Yukawa** empuja $\lambda$ al límite superior, los datos piden algo sin escala (curvas planas). Una fuerza de corto alcance como la fuerte no puede hacer este trabajo hoy.

**Limitaciones del notebook:**
- Aproximación esférica para discos; no se ajustan la distancia ni la inclinación.
- La fórmula de Verlinde vale para materia aislada y esférica.
- El modelo ER es un *ansatz* lineal mínimo. Una variante natural es la cuadrática: $I(\mathbf{x}:\mathbf{y}) \propto \rho_b(\mathbf{x})\rho_b(\mathbf{y})$, que daría $\rho_{ER} \propto \rho_b \cdot (K * \rho_b)$. También se podría añadir monogamia (saturación del entrelazamiento).
- Las curvas de rotación no bastan: el Cúmulo Bala, las lentes gravitacionales y el fondo cósmico de microondas son las pruebas siguientes.

**Referencias:** Maldacena y Susskind (2013); Van Raamsdonk (2010); Jacobson (2016); Verlinde (2017), *SciPost Phys.* 2, 016; Lelli, McGaugh y Schombert (2016), *AJ* 152, 157; Li et al. (2020), *ApJS* 247, 31 (ajustes de halos a SPARC); Brouwer et al. (2017) (prueba de Verlinde con lentes débiles).